# Functional neurotransmitter fingerprinting

In this tutorial, you will explore the functional neurotransmitter fingerprinting (FNTF) functionalities of Lacuna using the CLI.

**What you'll learn**:

- Fetch the neurotransmitter PET atlas and a functional connectome
- Prepare the atlas
- Compute NT scores weighted by functional connectivity
- Use ACE-enriched atlases for enhanced mapping

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-petersen/lacuna/blob/main/docs/tutorials/08-functional-neurotransmitter-fingerprinting.ipynb)

## Colab

Note: Colab provides limited computational resources. While these tutorials are designed to operate within those constraints, some Lacuna functionality cannot be fully demonstrated in this environment and requires access to higher-performance computing infrastructure.

Ignore this if you run this notebook locally.

## Setup

In [ ]:
# Install Lacuna from GitHub
!pip install git+https://github.com/m-petersen/lacuna

Get the tutorial data.

In [ ]:
# Get tutorial data
!lacuna tutorial /tmp/tutorial_bids --force

## Fetch data

FNTF requires two data sources:

1. **Neurotransmitter PET atlas** — PET receptor/transporter density maps from normative cohorts (from [OSF](https://osf.io/yz9mb/))
2. **Functional connectome** — Resting-state fMRI data from a normative cohort (e.g., [GSP1000](https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/ILXIKS))

For the GSP1000 download you need an API token. Register on [Harvard Dataverse](https://dataverse.harvard.edu/), click on your username and then API Token.

Here, we fetch the test version via `--test-mode` to keep the tutorial lightweight.

In [ ]:
# Fetch NT atlas
!lacuna fetch ntatlas \
    --output-dir /tmp/ntatlas_data

In [ ]:
# Fetch functional connectome (test mode)
!lacuna fetch gsp1000 \
    --output-dir /tmp/gsp1000_data \
    --api-key $DATAVERSE_API_KEY \
    --test-mode \
    --skip-checksum \
    --no-keep-original

## Prepare the atlas

Before running the analysis, the raw PET maps need to be prepared. This step averages maps per target and z-scores the result.

In [ ]:
!lacuna prepare lntf \
    --source-dir /tmp/ntatlas_data \
    --cache-dir /tmp/ntatlas_cache

## Analysis

Functional neurotransmitter fingerprinting combines functional connectivity with neurotransmitter information. It:

1. Computes the functional connectivity map of the lesion using the normative fMRI connectome
2. Weights NT atlas values by the functional connectivity at each voxel
3. Scores each neurotransmitter target based on connectivity-weighted NT values

This answers: **what NT systems are functionally connected to the lesion?**

A high score for a given target indicates that brain regions functionally connected to the lesion are rich in that neurotransmitter, suggesting potential remote neurochemical effects.

Run the analysis.

In [ ]:
!lacuna run fntf \
    /tmp/tutorial_bids/ \
    /tmp/outputs_fntf/ \
    --connectome-name GSP1000 \
    --participant-label 01 \
    --mask-space MNI152NLin6Asym \
    --atlas-cache-dir /tmp/ntatlas_cache

List the outputs.

In [ ]:
!ls /tmp/outputs_fntf/sub-01/ses-01/anat/

## Filter by neurotransmitter system

Restrict the analysis to specific neurotransmitter systems using the `--targets` flag:

| Preset | Targets |
|--------|--------|
| `dopaminergic` | D1, D23, DAT, FDOPA |
| `serotonergic` | 5HT1a, 5HT1b, 5HT2a, 5HT4, 5HT6, 5HTT |
| `cholinergic` | VAChT, M1, A4B2 |
| `monoaminergic` | D1, D23, DAT, 5HT1a, 5HT1b, 5HT2a, 5HT4, 5HT6, 5HTT, NET |
| `all` | All available targets (default) |

In [ ]:
!lacuna run fntf \
    /tmp/tutorial_bids/ \
    /tmp/outputs_fntf_dopamine/ \
    --connectome-name GSP1000 \
    --participant-label 01 \
    --mask-space MNI152NLin6Asym \
    --atlas-cache-dir /tmp/ntatlas_cache \
    --targets dopaminergic

## ACE-enriched atlas

For enhanced mapping, you can use an ACE (Atlas Connectivity Enrichment) enriched atlas. ACE incorporates functional connectivity information into the NT atlas itself, creating connectivity-informed neurotransmitter maps.

To use ACE-enriched mapping, first prepare the enriched atlas, then pass the `--enriched` flag:

```bash
# Prepare ACE-enriched atlas (one-time)
lacuna prepare ace --connectome-name GSP1000

# Run with enriched atlas
lacuna run fntf \
    /bids/ /output/ \
    --connectome-name GSP1000 \
    --enriched \
    --ace-cache-dir /path/to/ace
```

Note: ACE preparation is computationally intensive and requires the full connectome.

## Run on multiple subjects

Lacuna supports processing multiple subjects within a single run. If the `--participant-label` flag is omitted, the pipeline processes all subjects in the BIDS dataset.

FNTF leverages the same vectorized batch processing as standard functional network mapping for efficient multi-subject computation.

In [ ]:
!lacuna run fntf \
    /tmp/tutorial_bids/ \
    /tmp/outputs_fntf_all/ \
    --connectome-name GSP1000 \
    --mask-space MNI152NLin6Asym \
    --atlas-cache-dir /tmp/ntatlas_cache

Collect results into a group-level table.

In [ ]:
!lacuna collect \
    /tmp/outputs_fntf_all/ \
    --pattern "*fntf*parcelstats*" \
    --output-dir /tmp/group_fntm/

In [ ]:
!ls /tmp/group_fntm/